# Model: ARIMA Rolling Backtest

Rolling-window backtest of `auto_arima` on monthly bookings ARR.

- **Training window:** 3 years (36 months)
- **Forecast horizon:** 12 months
- **Skip:** 1 month (if training ends Dec, first forecast is Feb)
- **Roll:** advance 1 month per iteration until forecast end reaches Aug 2026

In [1]:
%pip install pmdarima --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\jash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

ModuleNotFoundError: No module named 'snowflake.snowpark'

## 1. Load Monthly Bookings Data

In [ ]:
query = """
WITH
fx AS (
    SELECT CURRENCY_CODE AS currency, TO_CHAR(STARTING_DATE, 'YYYY-MM') AS month_key, 1 / EXCHANGE_RATE AS rate
    FROM PROD_PROVISIONING.BUSINESS_CENTRAL.CURRENCY_EXCHANGE_RATE
),
nm(src, tgt) AS (
    SELECT * FROM VALUES
        ('Community Team','Community'),('APAC - Australia','Australia'),('APAC - Asia','Asia'),
        ('Retail - Enterprise','Retailer - Enterprise'),('Retail - Mid Market','Retailer - Mid Market'),
        ('SS - Analytics Ent','SS - Analytics Enterprise'),('SS - Analytics Mid','SS - Analytics Mid Market')
    AS t(src, tgt)
)
SELECT
    d.MONTH_CLOSED,
    ROUND(SUM(
        (CASE WHEN d.CURRENCY_CODE IN ('AUD','CAD','EUR')
              THEN d.BOOKINGS_ANNUALIZED_RECURRING_REVENUE / f.rate
              ELSE d.BOOKINGS_ANNUALIZED_RECURRING_REVENUE END)
        * os.SPLIT_PERCENTAGE / 100
    )) AS ARR_USD
FROM PROD_PROVISIONING.FINANCE_SHARE.DEALS_DATA d
JOIN PROD_PROVISIONING.SALESFORCE.OPPORTUNITY_SPLIT os
    ON os.OPPORTUNITY_ID = d.OPP_ID AND os.OPPORTUNITY_SPLIT_TYPE = 'Commissionable ARR'
JOIN PROD_PROVISIONING.SALESFORCE.OPPORTUNITY o
    ON o.OPPORTUNITY_ID = d.OPP_ID
LEFT JOIN fx f ON f.currency = d.CURRENCY_CODE AND f.month_key = d.MONTH_CLOSED
LEFT JOIN nm ON nm.src = os.EMPLOYEE_REPORTING_GROUP
WHERE d.DEAL_STAGE = 'Closed Won'
    AND d.MONTH_CLOSED >= '2020-01' AND d.MONTH_CLOSED <= '2026-08'
    AND COALESCE(o.OPPORTUNITY_TYPE, '') <> 'Rate Reduction'
    AND COALESCE(d.BOOKING, '') <> 'Other'
    AND NOT (COALESCE(d.BOOKING, '') = '' AND d.BOOKINGS_ANNUALIZED_RECURRING_REVENUE < 0)
    AND os.EMPLOYEE_REPORTING_GROUP NOT IN
        ('Do Not Report', 'System Admin', 'Customer Success', 'Revenue Recovery')
GROUP BY 1
ORDER BY 1
"""

df = session.sql(query).to_pandas()
df['MONTH_CLOSED'] = df['MONTH_CLOSED'].astype(str)
print(f"Loaded {len(df)} months: {df['MONTH_CLOSED'].iloc[0]} to {df['MONTH_CLOSED'].iloc[-1]}")
df.head()

## 2. Rolling Window Backtest

In [ ]:
import pandas as pd
import numpy as np
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

# Build a clean monthly series indexed by period
series = df.set_index('MONTH_CLOSED')['ARR_USD'].astype(float)
series.index = pd.PeriodIndex(series.index, freq='M')
series = series.sort_index()

TRAIN_MONTHS = 36    # 3 years
FORECAST_MONTHS = 12 # 1 year ahead
SKIP = 1             # skip 1 month

results = []
last_period = pd.Period('2026-08', freq='M')

for i in range(len(series)):
    train_start_idx = i
    train_end_idx = i + TRAIN_MONTHS - 1

    if train_end_idx >= len(series):
        break

    train = series.iloc[train_start_idx:train_end_idx + 1]
    train_end_period = train.index[-1]

    # forecast_start = train_end + 1 (as-of month, skipped) + 1 = train_end + 2
    forecast_start = train_end_period + 2
    forecast_end = forecast_start + FORECAST_MONTHS - 1

    if forecast_end > last_period:
        break

    # Actuals for the forecast window
    forecast_periods = pd.period_range(forecast_start, forecast_end, freq='M')
    actuals = series.reindex(forecast_periods)

    if actuals.isna().any():
        break

    # Fit auto_arima
    model = pm.auto_arima(
        train.values,
        seasonal=True,
        m=12,
        suppress_warnings=True,
        stepwise=True,
        error_action='ignore',
        max_p=3, max_q=3,
        max_P=2, max_Q=2,
        max_d=2, max_D=1
    )

    # Forecast SKIP + FORECAST_MONTHS steps, discard the skipped month
    fc = model.predict(n_periods=SKIP + FORECAST_MONTHS)
    preds = fc[SKIP:]

    # WMAPE
    abs_errors = np.abs(preds - actuals.values)
    wmape = abs_errors.sum() / actuals.values.sum() * 100

    # Worst month
    pct_errors = np.abs(preds - actuals.values) / actuals.values * 100
    worst_idx = pct_errors.argmax()
    worst_month = str(forecast_periods[worst_idx])
    worst_pct = round(pct_errors[worst_idx], 1)

    # ARIMA params
    order = model.order
    seasonal_order = model.seasonal_order
    params_str = f"({order[0]},{order[1]},{order[2]})({seasonal_order[0]},{seasonal_order[1]},{seasonal_order[2]})[{seasonal_order[3]}]"

    results.append({
        'train_start': str(train.index[0]),
        'train_end': str(train_end_period),
        'forecast_start': str(forecast_start),
        'forecast_end': str(forecast_end),
        'wmape_pct': round(wmape, 1),
        'worst_month': worst_month,
        'worst_month_err_pct': worst_pct,
        'arima_params': params_str
    })

    if len(results) % 5 == 0:
        print(f"  Window {len(results)}: train {train.index[0]}-{train_end_period}, WMAPE={wmape:.1f}%")

print(f"\nDone. {len(results)} backtest windows completed.")

## 3. Results Table

In [ ]:
results_df = pd.DataFrame(results)
print(f"Rolling Backtest Results: {len(results_df)} windows\n")
print(results_df.to_string(index=False))
print(f"\n--- Summary ---")
print(f"Mean WMAPE: {results_df['wmape_pct'].mean():.1f}%")
print(f"Median WMAPE: {results_df['wmape_pct'].median():.1f}%")
print(f"Best window:  {results_df.loc[results_df['wmape_pct'].idxmin(), 'forecast_start']} - {results_df.loc[results_df['wmape_pct'].idxmin(), 'forecast_end']} ({results_df['wmape_pct'].min():.1f}%)")
print(f"Worst window: {results_df.loc[results_df['wmape_pct'].idxmax(), 'forecast_start']} - {results_df.loc[results_df['wmape_pct'].idxmax(), 'forecast_end']} ({results_df['wmape_pct'].max():.1f}%)")

## 4. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# WMAPE over time
axes[0].bar(range(len(results_df)), results_df['wmape_pct'], color='steelblue', alpha=0.8)
axes[0].axhline(results_df['wmape_pct'].median(), color='red', linestyle='--', label=f"Median {results_df['wmape_pct'].median():.1f}%")
axes[0].set_xlabel('Window #')
axes[0].set_ylabel('WMAPE (%)')
axes[0].set_title('WMAPE by Rolling Window')
axes[0].legend()

# ARIMA params frequency
param_counts = results_df['arima_params'].value_counts().head(10)
axes[1].barh(param_counts.index, param_counts.values, color='coral', alpha=0.8)
axes[1].set_xlabel('Count')
axes[1].set_title('Most Frequent ARIMA Parameter Selections')

plt.tight_layout()
plt.show()

## 5. Save Results to Snowflake

In [ ]:
snow_df = session.create_dataframe(results_df)
snow_df.write.mode('overwrite').save_as_table('PROD_PROVISIONING.FINANCE_SHARE.MKT_ARIMA_BACKTEST_RESULTS')
print('Saved to PROD_PROVISIONING.FINANCE_SHARE.MKT_ARIMA_BACKTEST_RESULTS')